In [15]:
from IPython.display import Markdown, display
title = "NIRS Analysis"
authors = "Hugo Duchemin"
display(Markdown(f"# {title}"))
display(Markdown(f"by {authors}"))

# NIRS Analysis

by Hugo Duchemin

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the data & key informations

In [17]:
from snirf import Snirf
import pandas as pd
import numpy as np

sub = 'sub-001'
ses = 'ses-001'
path = f'/Users/hugoninho/Desktop/DigiMove/1st_bloc/Python-R-Git/Python /Data_Project_Valoxy/data/VALOXY-bids/{sub}/{ses}/nirs/'

def load_snirf_to_df(filepath, device_name):
    # Load the SNIRF file
    snirf_obj = Snirf(filepath)
    
    # Extract data from the first 'data' block
    # snirf_obj.nirs[0].data[0].dataTimeSeries is a numpy array (Time x Channels)
    data = snirf_obj.nirs[0].data[0].dataTimeSeries
    time = snirf_obj.nirs[0].data[0].time
    
    # Create DataFrame
    df = pd.DataFrame(data)
    df['time'] = time
    df['device'] = device_name
    return df

# Loading the files correctly
Semaxone_df = load_snirf_to_df(path + f'{sub}_{ses}_task-isometric_recording-semaxone_nirs.snirf', 'semaxone')
Portalite_df = load_snirf_to_df(path + f'{sub}_{ses}_task-isometric_recording-portalite_nirs.snirf', 'portalite')

# Concatenate if needed
nirs_data = pd.concat([Semaxone_df, Portalite_df], ignore_index=True)

print(nirs_data.head())

        0       1        2        3        4        5        6        7  \
0  9537.0  9427.0  13793.0  11929.0  12980.0  12377.0  16783.0  15970.0   
1  9550.0  9420.0  13785.0  11930.0  12994.0  12367.0  16773.0  15972.0   
2  9552.0  9416.0  13793.0  11934.0  13001.0  12360.0  16785.0  15976.0   
3  9554.0  9431.0  13783.0  11936.0  13003.0  12383.0  16774.0  15979.0   
4  9541.0  9435.0  13792.0  11938.0  12985.0  12387.0  16784.0  15984.0   

           8          9         10         11       12      13  time    device  
0  1024393.0  2651231.0  1307491.0  3783850.0  21308.0  8544.0  0.00  semaxone  
1  1024709.0  2653004.0  1307942.0  3786503.0  21310.0  8546.0  0.01  semaxone  
2  1024435.0  2651443.0  1307578.0  3784207.0  21310.0  8546.0  0.02  semaxone  
3  1024751.0  2653226.0  1308018.0  3786831.0  21313.0  8546.0  0.03  semaxone  
4  1024474.0  2651703.0  1307632.0  3784541.0  21314.0  8545.0  0.04  semaxone  


### **Observation** : This output allow us to understand the number of channels of each sensors, to check if no one is missing in the data set. 

# 2.Signal conversion & Pre-processing / Filtering

### **Observation** : To do that we have three different steps : raw intensity (sensor's output) --> optical density (logarithmic transformation) --> hemoglobin concentration (physiological)

In [53]:
import mne 
import matplotlib.pyplot as plt
import os

# --- Paramètres de la session ---
sub = "002"
ses = "002"
device = "portalite"  # Change to "portalite" if you want to analyze the portalite data instead

mne.viz.set_browser_backend("qt")

# --- Chemin unique vers le SNIRF ---
base_path = f"/Users/hugoninho/Desktop/DigiMove/1st_bloc/Python-R-Git/Python /Data_Project_Valoxy/data/VALOXY-bids/sub-{sub}/ses-{ses}"
path_snirf = os.path.join(base_path, "nirs", f"sub-{sub}_ses-{ses}_task-isometric_recording-{device}_nirs.snirf")

# --- Chargement des données ---
raw = mne.io.read_raw_snirf(path_snirf, preload=True)

# --- EXTRACTION INTERNE DES TRIGGERS ---
# On récupère les événements via find_events (qui scanne les canaux stim du SNIRF)
# Si find_events échoue, MNE basculera sur les annotations internes si elles existent
markers = mne.Annotations(
    onset=events["onset"].values,
    duration=events["duration"].values,
    description=events["trial_type"].values
    )
raw.set_annotations(markers)

# --- Pipeline de prétraitement ---

# 1. Intensité -> Densité Optique
raw_od = mne.preprocessing.nirs.optical_density(raw)

# 2. Densité Optique -> Concentrations (Haemoglobin)
# DPF fixé à 4 (ppf = 0.25) selon Pirovano et al. (2021)
raw_hb = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=4)

# 3. Filtrage (Butterworth 4ème ordre, 0.01 - 4 Hz)
raw_hb_filt = raw_hb.copy().filter(
    l_freq=0.01,
    h_freq=4,
    method="iir",
    iir_params=dict(order=4, ftype="butter"),
)

# --- Visualisation ---
duration = raw.times[-1]

# On affiche tout. Appuie sur 'a' dans la fenêtre si les triggers ne sont pas colorés.
raw_hb_filt.plot(
    duration=duration, 
    n_channels=len(raw_hb_filt.ch_names), 
    clipping=None,
    title=f"SNIRF Internal Data - Sub {sub} ({device})"
)

plt.show()

Loading /Users/hugoninho/Desktop/DigiMove/1st_bloc/Python-R-Git/Python /Data_Project_Valoxy/data/VALOXY-bids/sub-002/ses-002/nirs/sub-002_ses-002_task-isometric_recording-portalite_nirs.snirf
Found jitter of 0.000000% in sample times.


Reading 0 ... 22410  =      0.000 ...  2240.945 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.01 - 4 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.01, 4.00 Hz: -6.02, -6.02 dB



Channels marked as bad:
none


# 3. Feature extraction & Intra-Individual Analysis

In [22]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import os

# --- STEP 3: FEATURE EXTRACTION (INDIVIDUAL ANALYSIS) ---

# 1. Sélection manuelle
target_sub = "003"
target_ses = "001"

# 2. Chemin de sauvegarde
output_dir = r'/Users/hugoninho/Desktop/DigiMove/1st_bloc/Python-R-Git/Python /Data_Project_Valoxy/results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Paramètres
duration = 10.0
features_list = []

print(f"\n📊 Analyse Statistique en cours : Participant {target_sub}, Session {target_ses}")

# --- CORRECTION ICI : Recherche de tous les triggers de contraction ---
# On récupère tous les noms d'événements qui contiennent "Contraction"
all_event_names = filtered_events['trial_type'].unique()
contraction_events = [e for e in all_event_names if "Contraction" in e]

# On trie pour avoir un ordre logique (20, 40, 60 puis Rep1, Rep2...)
contraction_events.sort()

for event_name in contraction_events:
    # Extraction de l'intensité et du numéro de rep depuis le nom du trigger
    # Exemple : "Contraction_60_Rep1" -> Intensity: 60, Rep: 1
    parts = event_name.split('_')
    intensity_label = f"Block_{parts[1]}" # Crée "Block_60"
    rep_number = parts[2].replace('Rep', '') # Crée "1"

    onsets = filtered_events.loc[filtered_events['trial_type'] == event_name, 'onset'].values
    
    for t_start in onsets:
        t_end = t_start + duration
        
        try:
            start_idx, stop_idx = raw_hb_filt.time_as_index([t_start, t_end])
            data_segment, _ = raw_hb_filt[:, start_idx:stop_idx]
            
            for ch_idx, ch_name in enumerate(raw_hb_filt.ch_names):
                segment = data_segment[ch_idx]
                
                features_list.append({
                    'Subject': target_sub,
                    'Session': target_ses,
                    'Intensity': intensity_label,
                    'Repetition': rep_number,
                    'Channel': ch_name,
                    'Mean': np.mean(segment),
                    'STD': np.std(segment),
                    'Skewness': skew(segment),
                    'Kurtosis': kurtosis(segment)
                })
        except Exception as e:
            print(f"❌ Erreur sur {event_name} : {e}")

# 5. Sauvegarde
if features_list:
    df_current = pd.DataFrame(features_list)
    filename = f"results_S{target_sub}_SES{target_ses}_Semaxone.csv"
    full_path = os.path.join(output_dir, filename)
    
    df_current.to_csv(full_path, index=False)

    print("-" * 40)
    print(f"✅ Terminé ! {len(df_current)} lignes extraites (5 reps x 3 intensités x nb_canaux).")
    print(f"📍 Sauvegardé sous : {filename}")
else:
    print("❌ Aucun trigger 'Contraction' trouvé dans les événements.")


📊 Analyse Statistique en cours : Participant 003, Session 001
----------------------------------------
✅ Terminé ! 90 lignes extraites (5 reps x 3 intensités x nb_canaux).
📍 Sauvegardé sous : results_S003_SES001_Semaxone.csv


This section extracts key statistical features from the filtered NIRS signal (HbO/HbR) for each 10-second contraction at **20%, 40%, and 60% MVC**.

### 🔍 Intra-Individual Metrics
For each participant and session, we compute:
* **Mean & STD:** Magnitude and stability of the hemodynamic response.
* **Skewness & Kurtosis:** Distribution and shape of the signal (identifying drifts or abrupt changes).

> **Interactive Step:** Run this cell and input the **Subject ID** and **Session ID** to generate a participant-specific `.csv` file.


In [20]:
import pandas as pd
import glob
import os
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Lister tous les fichiers CSV créés précédemment
path = 'results_csv' # Le dossier où tu as sauvé tes fichiers
all_files = glob.glob(os.path.join(path, "*.csv"))

# 2. Fusionner tous les fichiers en un seul DataFrame
li = []
for filename in all_files:
    df = pd.read_csv(filename, index=False)
    li.append(df)

df_group = pd.concat(li, axis=0, ignore_index=True)

print(f"✅ Fusion terminée : {len(all_files)} fichiers combinés.")
print(f"Nombre total de lignes : {len(df_group)}")

# 3. Calculer les statistiques descriptives du GROUPE
# On groupe par Intensité et on calcule la moyenne et l'écart-type de tout le monde
group_stats = df_group.groupby(['Intensity']).agg({
    'Mean': ['mean', 'std'],
    'Skewness': ['mean', 'std'],
    'Kurtosis': ['mean', 'std']
}).reset_index()

print("\n--- STATISTIQUES DE GROUPE (Semaxone) ---")
print(group_stats)

ValueError: No objects to concatenate

Channels marked as bad:
none


In [ ]:
# Visualisation de la Mean HbO par intensité pour tout le groupe
plt.figure(figsize=(10, 6))
sns.barplot(data=df_group, x='Intensity', y='Mean', capsize=.1)
plt.title('Réponse Hémodynamique Moyenne par Intensité (Groupe N=3)')
plt.ylabel('Concentration HbO (M)')
plt.show()

### 🎯 Objectives of Group Analysis
The transition from individual to group data allows us to assess:
* **Session Stability:** Evaluating the **Test-Retest reliability** by comparing physiological responses between Session 1 and Session 2.
* **Intensity Discrimination:** Verifying if the system can statistically distinguish between **20%, 40%, and 60% MVC** workloads.
* **Consistency:** Ensuring that hemodynamic trends (HbO/HbR) remain coherent across the entire cohort despite inter-individual variability.


### **References**
--------------------------

-  Pirovano, I., Porcelli, S., Re, R., Spinelli, L., Contini, D., Marzorati, M., & Torricelli, A. (2021). Effect of adipose tissue thickness and tissue optical properties on the differential pathlength factor estimation for NIRS studies on human skeletal muscle. Biomedical Optics Express, 12(1), 571. https://doi.org/10.1364/BOE.412447

